# 合成双模型分交叉与保存后回放
明确为合成示例，不代表真实模型效果。X 高分低风险，Y 高分高风险。全部图表来自聚合快照，加载后不用原样本。

In [1]:
from pathlib import Path
import json
import numpy as np
import polars as pl
from mars.analysis import cross_scores, show_score_matrix, get_score_cell, evaluate_score_policy, write_score_cross_html
from mars.reporting import load_report
rng = np.random.default_rng(20261001)
n = 240
latent = rng.normal(size=n)
x = -latent + rng.normal(scale=0.4, size=n)
y = 1 / (1 + np.exp(-latent-rng.normal(size=n)))
x[::41] = np.nan
y[::29] = np.nan
bad = (rng.random(n) < 1/(1+np.exp(-latent))).astype(float)
late = (rng.random(n) < 0.2).astype(float)
bad[::7] = np.nan
late[::3] = np.nan
df = pl.DataFrame({'score_main':x, 'prob_aux':y, 'bad':bad, 'late':late, 'split':['TEST']*160+['OOT']*80})
reference = pl.DataFrame({'score_main':rng.normal(size=800), 'prob_aux':rng.uniform(size=800)})
report = cross_scores(df, score_x='score_main', score_y='prob_aux', targets=['bad','late'],
    score_directions={'score_main':'lower_risk','prob_aux':'higher_risk'}, binning_reference=reference,
    group_col='split', probability_scores=['prob_aux'], min_observed=15,
    feature_metadata={'score_main':{'display_name':'主模型分','data_source':'main'},'prob_aux':{'display_name':'辅助概率','data_source':'aux'}},
    business_context={'dataset_id':'synthetic-notebook','sample_unit':'synthetic row'})
output = Path('examples/output/notebook-score-cross')
output.mkdir(parents=True, exist_ok=True)
report.save(output/'score-cross.marsreport', overwrite=True)
restored = load_report(output/'score-cross.marsreport')
del df, report
show_score_matrix(restored, filters={'target':'bad','group':'OOT'})

,b0,b1,b2,b3,b4,TOTAL
b4,0.00% | n=8 | low_sample,16.67% | n=6 | low_sample,0.00% | n=1 | low_sample,— | n=0 | empty,— | n=0 | empty,6.25%
b3,33.33% | n=3 | low_sample,33.33% | n=3 | low_sample,0.00% | n=3 | low_sample,25.00% | n=4 | low_sample,— | n=0 | empty,21.43%
b2,33.33% | n=3 | low_sample,66.67% | n=3 | low_sample,100.00% | n=3 | low_sample,100.00% | n=1 | low_sample,25.00% | n=4 | low_sample,57.14%
b1,— | n=0 | empty,50.00% | n=4 | low_sample,66.67% | n=3 | low_sample,0.00% | n=3 | low_sample,50.00% | n=4 | low_sample,42.86%
b0,— | n=0 | empty,— | n=0 | empty,100.00% | n=1 | low_sample,100.00% | n=2 | low_sample,100.00% | n=5 | low_sample,100.00%
TOTAL,14.29%,35.29%,54.55%,40.00%,64.29%,39.71%


## 同 X 分段差值与可复查证据
差值以百分点展示；低样本保留真实值。

In [2]:
show_score_matrix(restored, filters={'target':'bad','group':'OOT'}, metric='delta_vs_row', color_range=(-0.2,0.2))

,b0,b1,b2,b3,b4,TOTAL
b4,-6.25pp | n=8 | low_sample,10.42pp | n=6 | low_sample,-6.25pp | n=1 | low_sample,— | n=0 | empty,— | n=0 | empty,6.25%
b3,11.90pp | n=3 | low_sample,11.90pp | n=3 | low_sample,-21.43pp | n=3 | low_sample,3.57pp | n=4 | low_sample,— | n=0 | empty,21.43%
b2,-23.81pp | n=3 | low_sample,9.52pp | n=3 | low_sample,42.86pp | n=3 | low_sample,42.86pp | n=1 | low_sample,-32.14pp | n=4 | low_sample,57.14%
b1,— | n=0 | empty,7.14pp | n=4 | low_sample,23.81pp | n=3 | low_sample,-42.86pp | n=3 | low_sample,7.14pp | n=4 | low_sample,42.86%
b0,— | n=0 | empty,— | n=0 | empty,0.00pp | n=1 | low_sample,0.00pp | n=2 | low_sample,0.00pp | n=5 | low_sample,100.00%
TOTAL,14.29%,35.29%,54.55%,40.00%,64.29%,39.71%


In [3]:
detail = get_score_cell(restored, 'b2','b2',filters={'target':'bad','group':'OOT'})
detail['page']['data'].to_pandas()

,group,period,target,x_bin,y_bin,x_risk_rank,y_risk_rank,sample_count,observed_sample_count,bad_sample_count,...,status,weighted_ci_status,unweighted_ci_status,scope_sample_count,overall_bad_rate,sample_share,lift_vs_overall,row_bad_rate,delta_vs_row,lift_vs_row
0,OOT,Total,bad,b2,b2,3,3,3,3,3,...,low_sample,not_requested,valid,80,0.397059,0.0375,2.518519,0.571429,0.428571,1.75


In [4]:
restored.get_table('cells', filters={'target':'bad','group':'OOT','x_bin':'b2'}, sort_by='y_risk_rank', columns=['y_bin','bad_rate','observed_sample_count','delta_vs_row','status']).to_pandas()

,y_bin,bad_rate,observed_sample_count,delta_vs_row,status
0,b0,0.333333,3,-0.238095,low_sample
1,b1,0.666667,3,0.095238,low_sample
2,b2,1.000000,3,0.428571,low_sample
3,b3,1.000000,1,0.428571,low_sample
4,b4,0.250000,4,-0.321429,low_sample
5,invalid,NaN,0,NaN,empty
6,missing,NaN,0,NaN,empty


## 历史规则比较
留存人数不同，不称为等覆盖或上线效果。缺失依赖轴默认拒绝。未列出的 staircase X 段拒绝。

In [5]:
baseline = {'type':'x_only','x_max_risk_rank':3}
rules = [baseline, {'type':'and','x_max_risk_rank':3,'y_max_risk_rank':3},
         {'type':'or','x_max_risk_rank':3,'y_max_risk_rank':3},
         {'type':'staircase','steps':{'b4':{'action':'accept','y_max_risk_rank':4},'b3':{'action':'accept','y_max_risk_rank':2},'b2':{'action':'reject'}}}]
policies = [evaluate_score_policy(restored, r, baseline=baseline) for r in rules]
policies[1].get_table('changes',filters={'target':'bad','group':'OOT'}).to_pandas()

,target,group,period,candidate_retained_count,candidate_retained_share,candidate_bad_rate,baseline_retained_count,baseline_retained_share,baseline_bad_rate,retained_count_delta,retained_share_delta,bad_rate_delta
0,bad,OOT,Total,41,0.5125,0.272727,54,0.675,0.272727,-13,-0.1625,0.0


In [6]:
write_score_cross_html(restored, output/'score-cross.html',policy_reports=policies)
restored.write_excel(str(output/'score-cross.xlsx'))
context = restored.to_ai_context(queries={'cells':{'filters':{'target':'bad','group':'OOT','x_bin':'b2'},'limit':5}},max_chars=16000)
assert json.loads(context)['evidence'][0]['report_id'] == restored.report_id
print('Offline HTML, Excel and portable evidence:', output)

Offline HTML, Excel and portable evidence: examples\output\notebook-score-cross
